# Same/Different Target Identity and Category-Pair Effects on LWS Probability (Q5, Q6)

- **Q5**: is the current miss's icon the *same* icon as the most recently identified target, or a
  *different* one?
- **Q6**: does the specific *pair* of (hit_category, miss_category) matter, beyond same/different? Restricted
  to visits with exactly one prior hit (so "the" recent hit is unambiguous) and to cells with >= 10 visits
  (see `ssm.py::build_category_pair_table` - the plan doc's judgment call on Q6 sparsity).

In [1]:
import os

from analysis.helpers.r_bridge import setup_rpy2, to_r_dataframe, source_r, get_r_object, glmer_metrics, cached_fit
setup_rpy2()

import pytensor
pytensor.config.cxx = ""
import bambi as bmb
import arviz as az

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from analysis.ssm_and_ab import ssm
from analysis.ssm_and_ab.ssm import load_ssm_funnel, add_ssm_predictors, build_category_pair_table

pio.renderers.default = "notebook"      # "notebook" or "browser"

In [2]:
data, funnel, hits = load_ssm_funnel()
funnel = add_ssm_predictors(funnel, hits)
lag_data = funnel.loc[funnel["num_targets_found_before"] > 0].copy()
print(f"{len(lag_data)} visits with a prior hit")

3269 visits with a prior hit


### Q5 - same icon vs. different icon

Landing back on an *already-identified* icon is normally classified `TARGET_RETURN`, not a miss, so
`same_icon_as_last_hit == True` visits are expected to be rare and strongly skewed toward `is_lws == False`.
Check the cross-tab below before trusting the model - a near-empty/near-degenerate cell shows up as a huge
standard error in the frequentist fit, not as an error.

In [3]:
pd.crosstab(lag_data["same_icon_as_last_hit"], lag_data["is_lws"])

is_lws,False,True
same_icon_as_last_hit,,
False,1748,615
True,906,0


In [4]:
def _fit_ssm_same_icon_glmm():
    model_data = lag_data[["subject", "trial", "trial_category", "same_icon_as_last_hit", "is_lws"]].copy()
    to_r_dataframe(model_data, "dat")
    source_r(os.path.join(os.getcwd(), "..", "R", "ssm_same_icon_glmm.R"))
    return {name: glmer_metrics(get_r_object(name), name) for name in ["model_flat", "model_nested"]}


ssm_same_icon_result = cached_fit(
    os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_same_icon_glmm.pkl"), _fit_ssm_same_icon_glmm,
)
for name, m in ssm_same_icon_result.items():
    print(f"--- {name} --- AIC={m['aic']:.1f}  converged={m['converged']}")
    display(m["coefficients"])

same_icon_row = ssm_same_icon_result["model_flat"]["coefficients"].pipe(
    lambda df: df.loc[df["term"].str.contains("same_icon")]
)
if (same_icon_row["std._error"] > 10).any():
    print(
        "WARNING: same_icon_as_last_hit's SE is very large - likely quasi/complete separation (near-zero "
        "is_lws rate among same_icon_as_last_hit==True visits). Treat this as evidence the cell is too "
        "sparse for a trustworthy frequentist effect size, not as a precise estimate - see the Bayesian "
        "companion below, whose priors regularize this case."
    )

R callback write-console: Loading required package: Matrix
  
R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: package 'Matrix' was built under R version 4.5.3 
  
R callback write-console: 2:   
R callback write-console: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :  
R callback write-console: 
   
R callback write-console:  unable to evaluate scaled gradient
  
R callback write-console: 3:   
R callback write-console: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :  
R callback write-console: 
   
R callback write-console:  Model failed to converge: degenerate  Hessian with 1 negative eigenvalues
  See ?lme4::convergence and ?lme4::troubleshooting.
  
R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In vcov.merMod(object, use.hessian = use.hessian) :  
R callback write-c

--- model_flat --- AIC=2681.4  converged=True


,term,estimate,std._error,z_value,prz
0,(Intercept),-1.286033,0.097880,-13.138946,1.969520e-39
1,same_icon_as_last_hitTRUE,-18.508240,28.802367,-0.642594,5.204873e-01
2,trial_categoryBW,0.597326,0.120411,4.960723,7.023136e-07
3,trial_categoryNOISE,0.238756,0.114314,2.088596,3.674410e-02


--- model_nested --- AIC=2680.4  converged=False


,term,estimate,std._error,z_value,prz
0,(Intercept),-1.348853,0.100051,-13.481696,2.004450e-41
1,same_icon_as_last_hitTRUE,-18.405684,553.099206,-0.033277,9.734534e-01
2,trial_categoryBW,0.631750,0.126332,5.000715,5.711816e-07
3,trial_categoryNOISE,0.257415,0.120100,2.143348,3.208518e-02


### Q6 - hit-category x miss-category

In [23]:
cat_table = build_category_pair_table(funnel, hits, min_cell_n=10)
print(f"{int(cat_table['sufficient_n'].sum())} / {len(cat_table)} cells have >= 10 visits")

lws_rate = cat_table.pivot(index="hit_category", columns="miss_category", values="lws_rate")
lws_rate.index = lws_rate.index.astype(lws_rate.columns.dtype)
lws_rate = lws_rate.reindex(columns=sorted(lws_rate.columns), index=sorted(lws_rate.index)).sort_index()

n_visits = cat_table.pivot(index="hit_category", columns="miss_category", values="n_visits")
n_visits.index = n_visits.index.astype(n_visits.columns.dtype)
n_visits = n_visits.reindex(columns=sorted(n_visits.columns), index=sorted(n_visits.index)).sort_index()

fig = go.Figure(go.Heatmap(
    z=lws_rate.values, x=lws_rate.columns.tolist(), y=lws_rate.index.tolist(),
    colorscale="Reds", colorbar=dict(title="P[LWS]"),
    text=n_visits.values, texttemplate="%{text}",
    hovertemplate="hit=%{y}<br>miss=%{x}<br>P[LWS]=%{z:.3f}<br>n=%{text}<extra></extra>",
))
fig.update_layout(
    title="P[LWS] by (Hit Category, Miss Category) - cell label = n visits",
    xaxis=dict(title=dict(text="Missed Target Category")),
    yaxis=dict(title=dict(text="Most Recently Hit Category")),
    template="plotly_white", width=700, height=600,
)
fig.show()

36 / 36 cells have >= 10 visits


In [6]:
def _fit_ssm_category_pair_glmm():
    joined = ssm.join_single_prior_hit_category(funnel, hits)
    sufficient = cat_table.loc[cat_table["sufficient_n"], ["hit_category", "miss_category"]]
    model_data = joined.merge(sufficient, on=["hit_category", "miss_category"], how="inner")[
        ["subject", "trial", "hit_category", "miss_category", "is_lws"]
    ]
    to_r_dataframe(model_data, "dat")
    source_r(os.path.join(os.getcwd(), "..", "R", "ssm_category_pair_glmm.R"))
    return {name: glmer_metrics(get_r_object(name), name) for name in ["model_flat", "model_nested"]}


ssm_category_pair_result = cached_fit(
    os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_category_pair_glmm.pkl"), _fit_ssm_category_pair_glmm,
)
for name, m in ssm_category_pair_result.items():
    print(f"--- {name} --- AIC={m['aic']:.1f}  converged={m['converged']}")
ssm_category_pair_result["model_flat"]["coefficients"]

R callback write-console: boundary (singular) fit: see help('isSingular')
  
R callback write-console: boundary (singular) fit: see help('isSingular')
  


Random effect variances not available. Returned R2 does not account for random effects.


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: Can't compute random effect variances. Some variance components equal
  zero. Your model may suffer from singularity (see `?lme4::isSingular`
  and `?performance::check_singularity`).
  Decrease the `tolerance` level to force the calculation of random effect
  variances, or impose priors on your random effects parameters (using
  packages like `brms` or `glmmTMB`). 
  


--- model_flat --- AIC=2386.5  converged=True
--- model_nested --- AIC=2388.5  converged=False


,term,estimate,std._error,z_value,prz
0,(Intercept),-1.252267,0.155259,-8.065689,7.282412e-16
1,hit_categoryANIMAL_OTHER,0.195615,0.210309,0.930131,3.523033e-01
2,hit_categoryHUMAN_FACE,0.018200,0.202160,0.090028,9.282648e-01
3,hit_categoryHUMAN_OTHER,0.283423,0.193696,1.463237,1.434027e-01
4,hit_categoryOBJECT_HANDMADE,0.015020,0.194823,0.077094,9.385486e-01
5,hit_categoryOBJECT_NATURAL,0.424279,0.182689,2.322408,2.021095e-02
6,miss_category.L,-0.526712,0.133526,-3.944626,7.992473e-05
7,miss_category.Q,-0.127854,0.127820,-1.000265,3.171821e-01
8,miss_category.C,0.063770,0.128458,0.496429,6.195918e-01
9,miss_category^4,0.031277,0.126368,0.247510,8.045138e-01


In [7]:
1/0

ZeroDivisionError: division by zero

### Bayesian companions (Q5, Q6)

Q5's frequentist fit above is expected to show near/complete separation. A Bayesian fit with
weakly-informative default priors regularizes exactly this case, so it may be the more trustworthy estimate
for Q5 specifically.

In [ ]:
bayes_data_q5 = lag_data[["subject", "trial", "trial_category", "same_icon_as_last_hit", "is_lws"]].copy()
bayes_cache_q5 = os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_same_icon_bayes_idata.nc")
bayes_model_q5 = bmb.Model(
    "is_lws ~ same_icon_as_last_hit + trial_category + (1|subject/trial)", bayes_data_q5, family="bernoulli",
)
if os.path.exists(bayes_cache_q5):
    idata_q5 = az.from_netcdf(bayes_cache_q5)
    print(f"Loaded cached idata from {bayes_cache_q5}")
else:
    idata_q5 = bayes_model_q5.fit(
        draws=2000, tune=1000, chains=4, cores=1, target_accept=0.95, random_seed=42, progressbar=False,
    )
    os.makedirs(os.path.dirname(bayes_cache_q5), exist_ok=True)
    az.to_netcdf(idata_q5, bayes_cache_q5)
    print(f"Fit complete, saved to {bayes_cache_q5}")

az.summary(idata_q5, var_names=["Intercept", "same_icon_as_last_hit"])

In [ ]:
sufficient = cat_table.loc[cat_table["sufficient_n"], ["hit_category", "miss_category"]]
bayes_data_q6 = ssm.join_single_prior_hit_category(funnel, hits).merge(
    sufficient, on=["hit_category", "miss_category"], how="inner"
)[["subject", "trial", "hit_category", "miss_category", "is_lws"]]

bayes_cache_q6 = os.path.join(os.getcwd(), "..", "R", "_cache", "ssm_category_pair_bayes_idata.nc")
bayes_model_q6 = bmb.Model(
    "is_lws ~ hit_category + miss_category + (1|subject/trial)", bayes_data_q6, family="bernoulli",
)
if os.path.exists(bayes_cache_q6):
    idata_q6 = az.from_netcdf(bayes_cache_q6)
    print(f"Loaded cached idata from {bayes_cache_q6}")
else:
    idata_q6 = bayes_model_q6.fit(
        draws=2000, tune=1000, chains=4, cores=1, target_accept=0.95, random_seed=42, progressbar=False,
    )
    os.makedirs(os.path.dirname(bayes_cache_q6), exist_ok=True)
    az.to_netcdf(idata_q6, bayes_cache_q6)
    print(f"Fit complete, saved to {bayes_cache_q6}")

az.summary(idata_q6, var_names=["Intercept", "hit_category", "miss_category"])

## Q7 - literature-grounded extensions (not all implemented)

Ranked by how much new engineering each needs beyond what `ssm.py` already builds:

1. **Fixation-lag vs. time-lag dissociation of the blink** (`02_temporal_dynamics.ipynb`, already
   implemented above) - tests whether the "blink" is a fixation-count phenomenon or a genuine temporal one.
2. **Spatial proximity between the last-hit target and the current miss** - an inhibition-of-return /
   foraging account distinct from a pure temporal blink. Needs one more join (icon-to-icon distance between
   the two targets, using the existing `pixel_distance`/`px2deg` utilities in `utils/distances.py`) but no
   new data.
3. **Dose-response over `num_targets_found_before`** (0 vs. 1 vs. 2, not just Q1's binary split) - already
   representable with existing columns (`funnel["num_targets_found_before"]`), just a different model spec
   than Q1's binary contrast; a natural sensitivity check on Q1 rather than a new question.
4. **Individual-difference stability** - does a subject's overall SSM rate correlate with their overall d'
   or hit rate (`utils/sdt.py` already computes both)? Needs only a subject-level aggregation join, no new
   tables.
5. **Target-target similarity as a moderator** (Cain et al.'s distractor/target-similarity account) - would
   need a similarity metric between target categories/exemplars that doesn't currently exist in this
   codebase; the one item here that's a genuinely new data-engineering task, not proposed for immediate
   implementation.

Items 1-4 are cheap enough to fold into these three notebooks as secondary models/plots; item 5 is listed
for awareness only.